# FedTrace — 02: Full-Scale Data Fetch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/02_fetch.ipynb)

**Runtime:** CPU only  
**Estimated time:** ~60 min (first run) — resumable if interrupted  
**Raw checkpoints:** `data/raw/` — local to Colab only (gitignored; persist across crashes, reset on runtime restart)  
**Publishes to GitHub:** `data/outputs/02_fetch.json`, `data/findings/02_fetch.md`

**Driving questions:**
1. Does the structural USASpending match gap (concentrated in Agriculture, DHS, GSA, VA, Justice, Interior, OPM) hold at scale across all 13,407 cancelled contracts?
2. What is the raw financial record for all resolvable cancelled contracts — ceiling, obligated, outlays?
3. How many of the 12,377 directly-linked cancelled grants have a complete three-number record?

**Design:** fetch only — no join logic. Raw API responses saved to `data/raw/`. Each section is resumable: already-fetched records are skipped on re-run. Join and assembly in notebook 03.

Current findings: [`data/findings/02_fetch.md`](../data/findings/02_fetch.md).

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('02_fetch', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['raw_dir'].mkdir(parents=True, exist_ok=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import requests
import pandas as pd
import json
import re
import time
import urllib.parse
from tqdm.notebook import tqdm

DOGE_BASE = 'https://api.doge.gov'
USA_BASE  = 'https://api.usaspending.gov/api/v2'
BATCH     = 50    # USASpending search batch size
DELAY     = 0.25  # seconds between batches

CONTRACT_TYPES = ['A', 'B', 'C', 'D']
IDV_TYPES      = ['IDV_A', 'IDV_B', 'IDV_B_A', 'IDV_B_B', 'IDV_B_C', 'IDV_C', 'IDV_D', 'IDV_E']

USA_FIELDS = [
    'Award ID', 'Recipient Name', 'Award Amount', 'Total Outlays',
    'Description', 'Contract Award Type', 'Awarding Agency',
    'Period of Performance Start Date',
    'Period of Performance Current End Date',
    'generated_internal_id',
]

_RETRYABLE = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


def doge_fetch(endpoint, per_page=500, max_pages=None):
    records, page = [], 1
    while True:
        r = requests.get(f'{DOGE_BASE}{endpoint}', params={'per_page': per_page, 'page': page})
        r.raise_for_status()
        data = r.json()
        key = next(iter(data['result']))
        records.extend(data['result'][key])
        total_pages = data['meta']['pages']
        print(f'  page {page}/{total_pages} — {len(records):,}', end='\r')
        if page >= total_pages or (max_pages and page >= max_pages):
            break
        page += 1
        time.sleep(0.1)
    print()
    return pd.DataFrame(records)


def _usa_search(piids, award_type_codes):
    """Single USASpending search batch — returns list of result dicts."""
    payload = {
        'filters': {'award_ids': piids, 'award_type_codes': award_type_codes},
        'fields': USA_FIELDS,
        'page': 1, 'limit': BATCH,
    }
    for attempt in range(6):
        wait = 2 ** attempt
        try:
            r = requests.post(f'{USA_BASE}/search/spending_by_award/', json=payload, timeout=30)
            if r.status_code == 429:
                print(f'\n  Rate limited — waiting {wait}s...')
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()['results']
        except _RETRYABLE as e:
            print(f'\n  Connection error (attempt {attempt+1}/6) — waiting {wait}s... [{e}]')
            time.sleep(wait)
    raise RuntimeError('Max retries exceeded on USASpending search')


def get_idv_amounts(generated_internal_id):
    """Fetch child task order obligations and outlays for an IDV."""
    for attempt in range(4):
        try:
            r = requests.get(f'{USA_BASE}/idvs/amounts/{generated_internal_id}/', timeout=15)
            if r.status_code == 200:
                body = r.json()
                return (
                    body.get('child_award_total_obligation'),
                    body.get('child_award_total_outlay'),
                )
            return None, None
        except _RETRYABLE:
            time.sleep(2 ** attempt)
    return None, None


def load_jsonl_ids(path, id_field):
    """Return set of IDs already saved in a JSONL checkpoint file."""
    path = Path(path)
    if not path.exists():
        return set()
    ids = set()
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                ids.add(json.loads(line).get(id_field))
    return ids


def append_jsonl(path, records):
    """Append records to a JSONL file."""
    with open(path, 'a') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')

## 1. Load DOGE Data

In [ ]:
print('Fetching contracts...')
contracts = doge_fetch('/savings/contracts')
print(f'Contracts: {len(contracts):,}')

print('Fetching grants...')
grants = doge_fetch('/savings/grants')
print(f'Grants: {len(grants):,}')

# Classify PIIDs
contracts['piid_type'] = contracts['piid'].apply(
    lambda x: 'aggregate' if str(x).startswith('Multiple') or ' ' in str(x) else 'single'
)
single = contracts[contracts['piid_type'] == 'single'].copy()

# Duplicate PIIDs exist in the DOGE data — keep last occurrence for metadata lookup
dupes = single[single.duplicated('piid', keep=False)]
if len(dupes):
    print(f'Duplicate PIIDs in DOGE data: {single["piid"].duplicated().sum():,} rows ({single["piid"].nunique():,} unique PIIDs)')
single_deduped = single.drop_duplicates('piid', keep='last')

contracts_by_piid = single_deduped.set_index('piid').to_dict('index')
all_piids = single_deduped['piid'].tolist()
print(f'\nSingle-PIID contracts (unique): {len(all_piids):,}')

# Grant linkage — extract award IDs from USASpending links
def extract_award_id(url):
    if not url:
        return None
    m = re.search(r'/award/([^/?#]+)', str(url))
    return m.group(1) if m else None

grants['link_host'] = grants['link'].apply(
    lambda u: urllib.parse.urlparse(str(u)).netloc.lower() if u else ''
)
usa_grants = grants[grants['link_host'].str.contains('usaspending', na=False)].copy()
usa_grants['award_id'] = usa_grants['link'].apply(extract_award_id)
usa_grants = usa_grants[usa_grants['award_id'].notna()].copy()
print(f'Grants with direct USASpending link: {len(usa_grants):,}')

## 2. Contract Fetch — Two-Pass with Checkpointing

Each batch runs pass 1 (contract types A–D) then pass 2 (IDV types) for unmatched PIIDs.  
Results are written to JSONL immediately — re-running this cell resumes from where it left off.

In [ ]:
matched_path   = PATHS['raw_dir'] / '02_contracts_matched.jsonl'
unmatched_path = PATHS['raw_dir'] / '02_contracts_unmatched.jsonl'

already_matched   = load_jsonl_ids(matched_path, 'piid')
already_unmatched = load_jsonl_ids(unmatched_path, 'piid')
processed = already_matched | already_unmatched
remaining = [p for p in all_piids if p not in processed]

print(f'Total single-PIID contracts: {len(all_piids):,}')
print(f'  Already matched:   {len(already_matched):,}')
print(f'  Already unmatched: {len(already_unmatched):,}')
print(f'  Remaining:         {len(remaining):,}')

skipped_batches = 0

for i in tqdm(range(0, len(remaining), BATCH), desc='Contracts'):
    batch = remaining[i:i+BATCH]

    # Pass 1 — direct contract types
    try:
        p1_results = _usa_search(batch, CONTRACT_TYPES)
    except RuntimeError:
        print(f'\n  Skipping batch {i//BATCH} after exhausting retries')
        skipped_batches += 1
        continue
    p1_matched = {r['Award ID'] for r in p1_results}
    for r in p1_results:
        piid = r['Award ID']
        append_jsonl(matched_path, [{**contracts_by_piid.get(piid, {'piid': piid}), **r, 'record_type': 'contract'}])
    time.sleep(DELAY)

    # Pass 2 — IDV types for unmatched
    p1_unmatched = [p for p in batch if p not in p1_matched]
    if p1_unmatched:
        try:
            p2_results = _usa_search(p1_unmatched, IDV_TYPES)
        except RuntimeError:
            print(f'\n  Skipping IDV pass for batch {i//BATCH} after exhausting retries')
            skipped_batches += 1
            continue
        p2_matched = {r['Award ID'] for r in p2_results}
        for r in p2_results:
            piid = r['Award ID']
            append_jsonl(matched_path, [{**contracts_by_piid.get(piid, {'piid': piid}), **r, 'record_type': 'idv'}])
        for piid in p1_unmatched:
            if piid not in p2_matched:
                append_jsonl(unmatched_path, [{**contracts_by_piid.get(piid, {'piid': piid}), 'status': 'unmatched_both_passes'}])
        time.sleep(DELAY)

matched_total   = len(load_jsonl_ids(matched_path, 'piid'))
unmatched_total = len(load_jsonl_ids(unmatched_path, 'piid'))
print(f'\nContract fetch complete.')
print(f'  Matched:   {matched_total:,} ({matched_total/len(all_piids)*100:.1f}%)')
print(f'  Unmatched: {unmatched_total:,} ({unmatched_total/len(all_piids)*100:.1f}%)')
if skipped_batches:
    print(f'  Skipped batches (exhausted retries): {skipped_batches} — re-run cell to retry')

## 3. IDV Child Amounts

For each IDV in the matched set, fetch `child_award_total_obligation` and `child_award_total_outlay` from `/api/v2/idvs/amounts/{generated_internal_id}/`.  
Saved separately — join happens in notebook 03.

In [ ]:
idv_amounts_path = PATHS['raw_dir'] / '02_idv_amounts.jsonl'
already_fetched_gids = load_jsonl_ids(idv_amounts_path, 'generated_internal_id')

# Load all matched IDVs
idvs_to_fetch = []
with open(matched_path) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        if r.get('record_type') == 'idv':
            gid = r.get('generated_internal_id')
            if gid and gid not in already_fetched_gids:
                idvs_to_fetch.append({'gid': gid, 'piid': r.get('piid') or r.get('Award ID')})

print(f'IDVs already fetched: {len(already_fetched_gids):,}')
print(f'IDVs to fetch:        {len(idvs_to_fetch):,}')

for item in tqdm(idvs_to_fetch, desc='IDV amounts'):
    obl, out = get_idv_amounts(item['gid'])
    append_jsonl(idv_amounts_path, [{
        'generated_internal_id': item['gid'],
        'piid': item['piid'],
        'child_award_total_obligation': obl,
        'child_award_total_outlay': out,
    }])
    time.sleep(0.15)

print(f'IDV amounts complete: {len(load_jsonl_ids(idv_amounts_path, "generated_internal_id")):,} records')

## 4. Grant Fetch

Direct `/api/v2/awards/{id}/` lookup for each grant with a USASpending `link`.  
Resumable — already-fetched award IDs are skipped.

In [ ]:
grants_path = PATHS['raw_dir'] / '02_grants.jsonl'
already_fetched_aids = load_jsonl_ids(grants_path, 'award_id')

to_fetch = [
    (row['award_id'], row.to_dict())
    for _, row in usa_grants.iterrows()
    if row['award_id'] not in already_fetched_aids
]

print(f'Grants already fetched: {len(already_fetched_aids):,}')
print(f'Grants to fetch:        {len(to_fetch):,}')

errors = 0
for award_id, doge_row in tqdm(to_fetch, desc='Grants'):
    for attempt in range(4):
        try:
            r = requests.get(f'{USA_BASE}/awards/{award_id}/', timeout=15)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            if r.status_code == 200:
                d = r.json()
                record = {
                    **doge_row,
                    'award_id': award_id,
                    'recipient_name': d.get('recipient', {}).get('recipient_name'),
                    'total_obligation': d.get('total_obligation'),
                    'total_outlays': d.get('total_outlays'),
                    'awarding_agency': d.get('awarding_agency', {}).get('toptier_agency', {}).get('name'),
                    'description': d.get('description'),
                    'period_start': d.get('period_of_performance', {}).get('start_date'),
                    'period_end': d.get('period_of_performance', {}).get('end_date'),
                }
                append_jsonl(grants_path, [record])
            break
        except Exception:
            errors += 1
            time.sleep(1)
    time.sleep(DELAY)

fetched_total = len(load_jsonl_ids(grants_path, 'award_id'))
print(f'\nGrant fetch complete.')
print(f'  Fetched: {fetched_total:,} ({fetched_total/len(usa_grants)*100:.1f}% of linked grants)')
print(f'  Errors:  {errors:,}')

## 5. Coverage Report

In [ ]:
# Load all matched contracts for agency breakdown
matched_records = []
with open(matched_path) as f:
    for line in f:
        if line.strip():
            matched_records.append(json.loads(line))

unmatched_records = []
with open(unmatched_path) as f:
    for line in f:
        if line.strip():
            unmatched_records.append(json.loads(line))

matched_df   = pd.DataFrame(matched_records)
unmatched_df = pd.DataFrame(unmatched_records)

print(f'=== Contract Coverage ===')
print(f'Total single-PIID contracts: {len(all_piids):,}')
print(f'Matched:   {len(matched_df):,} ({len(matched_df)/len(all_piids)*100:.1f}%)')
print(f'  — contracts: {(matched_df["record_type"]=="contract").sum():,}')
print(f'  — IDVs:      {(matched_df["record_type"]=="idv").sum():,}')
print(f'Unmatched: {len(unmatched_df):,} ({len(unmatched_df)/len(all_piids)*100:.1f}%)')
print()

# Agency breakdown — unmatched vs matched
agency_breakdown = pd.DataFrame({
    'unmatched': unmatched_df['agency'].value_counts(),
    'matched':   matched_df['agency'].value_counts(),
}).fillna(0).astype(int)
agency_breakdown['unmatched_rate'] = (
    agency_breakdown['unmatched'] /
    (agency_breakdown['unmatched'] + agency_breakdown['matched']) * 100
).round(1)
print('Agency breakdown (sorted by unmatched rate):')
print(agency_breakdown.sort_values('unmatched_rate', ascending=False).head(15).to_string())
print()

grants_fetched = len(load_jsonl_ids(grants_path, 'award_id'))
print(f'=== Grant Coverage ===')
print(f'Directly-linked grants: {len(usa_grants):,}')
print(f'Fetched: {grants_fetched:,} ({grants_fetched/len(usa_grants)*100:.1f}%)')

## 6. Publish

In [ ]:
# Summary JSON
idv_amounts_fetched = len(load_jsonl_ids(idv_amounts_path, 'generated_internal_id'))

output = {
    'contracts': {
        'total_single_piid': len(all_piids),
        'matched': len(matched_df),
        'matched_pct': round(len(matched_df) / len(all_piids) * 100, 1),
        'matched_as_contract': int((matched_df['record_type'] == 'contract').sum()),
        'matched_as_idv': int((matched_df['record_type'] == 'idv').sum()),
        'unmatched': len(unmatched_df),
        'idv_amounts_fetched': idv_amounts_fetched,
    },
    'grants': {
        'directly_linked': len(usa_grants),
        'fetched': grants_fetched,
        'fetched_pct': round(grants_fetched / len(usa_grants) * 100, 1),
    },
    'raw_files': [
        'data/raw/02_contracts_matched.jsonl',
        'data/raw/02_contracts_unmatched.jsonl',
        'data/raw/02_idv_amounts.jsonl',
        'data/raw/02_grants.jsonl',
    ],
}

output_path = PATHS['outputs_dir'] / '02_fetch.json'
output_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

# Findings
top_unmatched = agency_breakdown.sort_values('unmatched_rate', ascending=False).head(10)
agency_lines = '\n'.join(
    f'  - {agency}: {row["unmatched"]} unmatched / {row["matched"]} matched ({row["unmatched_rate"]}%)'
    for agency, row in top_unmatched.iterrows()
)

findings_md = f"""# Findings — 02: Full-Scale Data Fetch

*Dataset: {len(all_piids):,} single-PIID contracts, {len(usa_grants):,} directly-linked grants.*  
*Methodology: `notebooks/02_fetch.ipynb`*

## Confirmed

- **Contract match rate at scale:** {len(matched_df):,}/{len(all_piids):,} ({len(matched_df)/len(all_piids)*100:.1f}%) resolved via two-pass USASpending lookup.
  - As direct contracts: {int((matched_df['record_type']=='contract').sum()):,}
  - As IDVs: {int((matched_df['record_type']=='idv').sum()):,}
- **Agency match gap — at scale:**
{agency_lines}
- **Grant fetch:** {grants_fetched:,}/{len(usa_grants):,} ({grants_fetched/len(usa_grants)*100:.1f}%) directly-linked grants fetched.

## Open

- Three-number record assembly and aggregate findings — notebook 03.
- Linkage path for grants with no `link` host — not addressed in this notebook.
"""

findings_path = PATHS['data_root'] / 'findings' / '02_fetch.md'
findings_path.parent.mkdir(parents=True, exist_ok=True)
findings_path.write_text(findings_md)
print(findings_md)

In [ ]:
from scripts.colab_utils import publish_artifacts

# Raw JSONL files are gitignored — too large for GitHub, persist in Colab across crashes.
# Only publish derived outputs: summary JSON and findings.
publish_artifacts(
    paths=[
        'data/outputs/02_fetch.json',
        'data/findings/02_fetch.md',
    ],
    message='Full-scale data fetch',
    repo_dir=REPO,
)